<a href="https://colab.research.google.com/github/FerhatPasha/Experiments-for-Frames/blob/main/stiefel_concentration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import quad
from scipy.special import gammainc
from tqdm.notebook import tqdm

# -----------------------------
# Parameters
# -----------------------------
d = 5
epsilon = 0.1

n_dense = np.arange(500, 1201, 20)
n_sparse = np.arange(1700, 7201, 500)
n_values = np.concatenate((n_dense, n_sparse))

num_trials = 3000
rng = np.random.default_rng(42)

# -----------------------------
# Compute K
# -----------------------------
def compute_K(n, d):
    """
    Uses the change of variables y = t^2.
    Then dy/sqrt(y) = 2 dt.
    """

    a = d / 2

    def integrand(t):
        F = gammainc(a, t**2)
        return 2 * (1 - F**n)

    integral_value, error = quad(
        integrand,
        0,
        np.inf,
        epsabs=1e-8,
        epsrel=1e-8,
        limit=200
    )

    K = (1 + d / (2*n)) * (1 / np.sqrt(2*n)) * integral_value - np.sqrt(d / n)

    return K

# -----------------------------
# Theoretical bound
# -----------------------------
def theoretical_bound_stiefel(n, d, epsilon):
    K = compute_K(n, d)

    if K >= epsilon:
        return np.nan, K

    bound = np.exp(- ((n - 2) * (epsilon - K)**2) / 8)

    return bound, K

# -----------------------------
# Sample from St(n,d)
# -----------------------------
def sample_stiefel(n, d, rng):
    """
    Samples a Haar-distributed element of St(n,d),
    represented as an n x d matrix with orthonormal columns.
    """
    G = rng.normal(size=(n, d))
    Q, R = np.linalg.qr(G, mode="reduced")

    # Sign correction for Haar consistency
    signs = np.sign(np.diag(R))
    signs[signs == 0] = 1
    Q = Q * signs

    return Q

# -----------------------------
# Monte Carlo estimate
# -----------------------------
def empirical_probability_stiefel(n, d, epsilon, num_trials, rng):
    count = 0
    threshold = np.sqrt(d / n)

    for _ in range(num_trials):
        X = sample_stiefel(n, d, rng)

        row_norms = np.linalg.norm(X, axis=1)
        max_row_norm = np.max(row_norms)

        if max_row_norm - threshold >= epsilon:
            count += 1

    return count / num_trials

# -----------------------------
# Run experiment
# -----------------------------
empirical_probs = []
theoretical_probs = []
K_values = []

for n in tqdm(n_values, desc="Running Stiefel experiment"):
    emp = empirical_probability_stiefel(n, d, epsilon, num_trials, rng)
    th, K = theoretical_bound_stiefel(n, d, epsilon)

    empirical_probs.append(emp)
    theoretical_probs.append(th)
    K_values.append(K)

empirical_probs = np.array(empirical_probs)
theoretical_probs = np.array(theoretical_probs)
K_values = np.array(K_values)

theoretical_probs_clipped = np.minimum(theoretical_probs, 1)

# -----------------------------
# Plot probability comparison
# -----------------------------
plt.figure(figsize=(10, 5.5))

plt.plot(
    n_values,
    empirical_probs,
    marker="o",
    markersize=4,
    linestyle="-",
    color="orange",
    label=r"Empirical $P(\max_i \|u_i\| - \sqrt{d/n} \geq \varepsilon)$"
)

plt.plot(
    n_values,
    theoretical_probs_clipped,
    linestyle="--",
    linewidth=2,
    color="purple",
    label="Theoretical bound"
)

plt.title(rf"Stiefel model: $P$ vs $n$ $(d={d}, \varepsilon={epsilon})$", fontsize=14)
plt.xlabel(r"$n$", fontsize=12)
plt.ylabel(r"$P$", fontsize=12)

plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# -----------------------------
# Plot K values
# -----------------------------
plt.figure(figsize=(10, 4.8))

plt.plot(
    n_values,
    K_values,
    marker="o",
    markersize=4,
    linestyle="-",
    color="black",
    label=r"$K$"
)

plt.axhline(
    epsilon,
    linestyle="--",
    linewidth=2,
    color="gray",
    label=r"$\varepsilon$"
)

plt.title(rf"$K$ vs $n$ $(d={d}, \varepsilon={epsilon})$", fontsize=14)
plt.xlabel(r"$n$", fontsize=12)
plt.ylabel(r"$K$", fontsize=12)

plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import quad
from scipy.special import gammainc
from tqdm.notebook import tqdm

# -----------------------------
# Parameters
# -----------------------------
d = 50
epsilon = 0.01

n_dense = np.arange(100000, 200001, 2000)
n_sparse = np.arange(300000, 800001, 100000)
n_values = np.concatenate((n_dense, n_sparse))

num_trials = 300
rng = np.random.default_rng(42)

# -----------------------------
# Compute K
# -----------------------------
def compute_K(n, d):
    """
    Computes

    K = (1 + d/(2n)) * 1/sqrt(2n)
        * integral_0^infty [1 - (gamma(d/2,y)/Gamma(d/2))^n] / sqrt(y) dy
        - sqrt(d/n)

    Uses the substitution y = t^2, so dy/sqrt(y) = 2 dt.
    """

    a = d / 2

    def integrand(t):
        F = gammainc(a, t**2)
        return 2 * (1 - F**n)

    integral_value, error = quad(
        integrand,
        0,
        np.inf,
        epsabs=1e-8,
        epsrel=1e-8,
        limit=300
    )

    K = (1 + d / (2*n)) * (1 / np.sqrt(2*n)) * integral_value - np.sqrt(d / n)

    return K

# -----------------------------
# Theoretical bound
# -----------------------------
def theoretical_bound_stiefel(n, d, epsilon):
    K = compute_K(n, d)

    if K >= epsilon:
        return np.nan, K

    bound = np.exp(- ((n - 2) * (epsilon - K)**2) / 8)

    return bound, K

# -----------------------------
# Sample from St(n,d)
# -----------------------------
def sample_stiefel(n, d, rng):
    """
    Samples a Haar-distributed element of St(n,d),
    represented as an n x d matrix with orthonormal columns.
    """
    G = rng.normal(size=(n, d))
    Q, R = np.linalg.qr(G, mode="reduced")

    # Sign correction for Haar consistency
    signs = np.sign(np.diag(R))
    signs[signs == 0] = 1
    Q = Q * signs

    return Q

# -----------------------------
# Monte Carlo estimate
# -----------------------------
def empirical_probability_stiefel(n, d, epsilon, num_trials, rng):
    count = 0
    threshold = np.sqrt(d / n)

    for _ in range(num_trials):
        X = sample_stiefel(n, d, rng)

        row_norms = np.linalg.norm(X, axis=1)
        max_row_norm = np.max(row_norms)

        if max_row_norm - threshold >= epsilon:
            count += 1

    return count / num_trials

# -----------------------------
# Run experiment
# -----------------------------
empirical_probs = []
theoretical_probs = []
K_values = []

for n in tqdm(n_values, desc="Running Stiefel experiment"):
    emp = empirical_probability_stiefel(n, d, epsilon, num_trials, rng)
    th, K = theoretical_bound_stiefel(n, d, epsilon)

    empirical_probs.append(emp)
    theoretical_probs.append(th)
    K_values.append(K)

empirical_probs = np.array(empirical_probs)
theoretical_probs = np.array(theoretical_probs)
K_values = np.array(K_values)

theoretical_probs_clipped = np.minimum(theoretical_probs, 1)

# -----------------------------
# Plot probability comparison
# -----------------------------
plt.figure(figsize=(10, 5.5))

plt.plot(
    n_values,
    empirical_probs,
    marker="o",
    markersize=4,
    linestyle="-",
    color="orange",
    label=r"Empirical $P(\max_i \|u_i\| - \sqrt{d/n} \geq \varepsilon)$"
)

plt.plot(
    n_values,
    theoretical_probs_clipped,
    linestyle="--",
    linewidth=2,
    color="purple",
    label="Theoretical bound"
)

plt.title(rf"Stiefel model: $P$ vs $n$ $(d={d}, \varepsilon={epsilon})$", fontsize=14)
plt.xlabel(r"$n$", fontsize=12)
plt.ylabel(r"$P$", fontsize=12)

plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# -----------------------------
# Plot K values
# -----------------------------
plt.figure(figsize=(10, 4.8))

plt.plot(
    n_values,
    K_values,
    marker="o",
    markersize=4,
    linestyle="-",
    color="black",
    label=r"$K$"
)

plt.axhline(
    epsilon,
    linestyle="--",
    linewidth=2,
    color="gray",
    label=r"$\varepsilon$"
)

plt.title(rf"$K$ vs $n$ $(d={d}, \varepsilon={epsilon})$", fontsize=14)
plt.xlabel(r"$n$", fontsize=12)
plt.ylabel(r"$K$", fontsize=12)

plt.grid(True, alpha=0.3)
plt.legend()
plt.show()